# 03 · Embeddings and the vector store\n\nChunks are embedded (Cohere Embed in live mode) and stored in Qdrant with a **dense** vector, a **BM25 sparse** vector and a payload (owner, document, page, section, text). Without API keys this notebook uses the offline hashing embedder: the steps are identical, the numbers are not meaningful.

In [1]:
import os, sys, uuid, json, asyncio
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "backend"))
os.chdir(ROOT / "backend")
from app.config import Settings
from app.services.container import build_container
_env = Settings()  # reads ../.env like the app does
LIVE = bool(_env.cohere_api_key and _env.anthropic_api_key)
settings = Settings(use_fake_providers=not LIVE, qdrant_url=None, qdrant_path=None)  # in-process Qdrant
c = build_container(settings)
print("MODE:", "LIVE (Cohere + Claude)" if LIVE else "FAKE providers - shows the mechanics only, not real quality")

MODE: FAKE providers - shows the mechanics only, not real quality


## 1. Ingest four corpus documents for user `alice`

In [2]:
from app.ingestion.pipeline import process_file
from app.clients.sparse import doc_sparse
from app.guardrails import scan_document_text
CORPUS = ROOT / "data" / "corpus"
await c.store.ensure_collection()
DOCS = {}
for name in ["employee_handbook_2026.pdf", "travel_expense_policy.docx", "it_security_standard.pdf", "remote_work_policy_2024.docx"]:
    res = process_file(name, (CORPUS / name).read_bytes(), document_id=str(uuid.uuid4()), settings=settings)
    dense = await c.embedder.embed_documents([ch.embed_text for ch in res.chunks])
    await c.store.upsert(res.chunks, dense, [doc_sparse(ch.embed_text) for ch in res.chunks], "alice",
                         [scan_document_text(ch.text) for ch in res.chunks])
    DOCS[name] = res
    print(f"{name:32s} {len(res.chunks):>2} chunks  vector dim={len(dense[0])}")

employee_handbook_2026.pdf        9 chunks  vector dim=256
travel_expense_policy.docx        4 chunks  vector dim=256


it_security_standard.pdf          3 chunks  vector dim=256
remote_work_policy_2024.docx      2 chunks  vector dim=256


## 2. Cosine similarity: question vs. every chunk\nThe top chunks should be the ones about sick leave.

In [3]:
from app.retrieval import cosine
q = "How many paid sick days do employees get?"
qv = await c.embedder.embed_query(q)
chunks = [ch for r in DOCS.values() for ch in r.chunks]
vecs = await c.embedder.embed_documents([ch.embed_text for ch in chunks])
for score, ch in sorted(((cosine(qv, v), ch) for v, ch in zip(vecs, chunks)), key=lambda t: -t[0])[:5]:
    print(f"{score:.3f}  {ch.filename:30s} {ch.section!s:28.28s} {ch.text[:60]!r}")

0.264  employee_handbook_2026.pdf     2 Leave > 2.2 Sick leave     'Employees are entitled to 12 paid sick days per year. A medi'
0.200  employee_handbook_2026.pdf     1 Introduction > 1.2 Probati 'New employees complete a probation period of 6 months. Notic'
0.192  employee_handbook_2026.pdf     2 Leave > 2.1 Annual leave   'Full-time employees receive 22 days of paid annual leave per'
0.158  remote_work_policy_2024.docx   Remote Work Policy (2024) >  'Employees may work remotely up to 2 days per week after comp'
0.150  employee_handbook_2026.pdf     5 Conduct                    'Gifts from clients or suppliers worth more than USD 75 must '


## 3. What Qdrant stores (one point)

In [4]:
hits = await c.store.search_dense(qv, "alice", limit=1)
print(json.dumps({k: v for k, v in hits[0].payload.items() if k != "text"}, indent=2))

{
  "user_id": "alice",
  "document_id": "0a130830-8a2b-4030-8a52-050d0c1f6310",
  "filename": "employee_handbook_2026.pdf",
  "position": 4,
  "page_start": 2,
  "page_end": 2,
  "section": "2 Leave > 2.2 Sick leave",
  "kind": "text",
  "chunking_config": "B",
  "suspicious": false
}


## 4. Per-user isolation: another user sees nothing

In [5]:
print("alice:", len(await c.store.search_dense(qv, "alice", limit=5)), "hits")
print("bob:  ", len(await c.store.search_dense(qv, "bob", limit=5)), "hits")

alice: 5 hits
bob:   0 hits


## 5. Deleting a document removes all of its vectors

In [6]:
doc = DOCS["remote_work_policy_2024.docx"]
did = doc.chunks[0].document_id
print("before:", await c.store.count_document("alice", did))
print("remaining after delete:", await c.store.delete_document("alice", did))

before: 2
remaining after delete: 0
